# A — Cleaning & Integration Pipeline

**Goal:** turn the three raw exports into trusted, time-aligned master tables. Run this notebook first. Raw files are never edited. All timestamps are standardized to Addis Ababa local time (EAT, UTC+03:00).

In [ ]:
from pathlib import Path
import sys, pandas as pd
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
from pipeline import load_raw, clean_tables, engineer_features, run_pipeline
train_raw,test_raw,weather_raw,events_raw=load_raw()
print('Raw shapes:', train_raw.shape, test_raw.shape, weather_raw.shape, events_raw.shape)
print('Train columns:', train_raw.columns.tolist())

## A1. Cleaning log
We audit mixed date formats, inconsistent zone/event labels, impossible values, missing values, duplicates, and malformed event intervals. Operational columns are retained for analysis but excluded from forecasting features.

In [ ]:
result=run_pipeline(run_models=False)
print(result['cleaning_log'].to_string(index=False))
print('Clean train:',result['train'].shape,'Clean test:',result['test'].shape)

## A2. Time and key standardization
Naive trip/event times are interpreted as local EAT. Weather timestamps carrying `Z` or an explicit UTC offset are converted to `Africa/Addis_Ababa`; naive weather timestamps are treated as local. The clock-shift diagnostic is saved to `reports/B2_1_clock_shift_test.csv`. Inspect it alongside the timestamp formats; correlation alone is not proof of a timezone.

In [ ]:
print('Canonical train zones:',sorted(result['train'].zone.unique()))
print('Weather hours:',result['weather'].pickup_hour.min(),'to',result['weather'].pickup_hour.max())
print(pd.read_csv(ROOT/'reports/A_join_audit.json').to_string(index=False) if False else (ROOT/'reports/A_join_audit.json').read_text())

## A3–A5. Join map, audit, and proof
The weather join is a **many-to-one left join** on normalized local hour; weather is deduplicated/aggregated to one row per hour first. Events are interval features: affected zone-hours from 2 hours before start through 2 hours after end are flagged. The agricultural/ride table remains the left table so missing support data does not delete a demand observation.

In [ ]:
print(pd.read_csv(ROOT/'reports/A7_integrity_checks.csv').to_string(index=False))
print('Master train columns:',pd.read_csv(ROOT/'data/processed/master_train.csv',nrows=1).columns.tolist())
print('Master test columns:',pd.read_csv(ROOT/'data/processed/master_test.csv',nrows=1).columns.tolist())

## A6. Feature engineering table
Calendar: hour, day-of-week, weekend, month, day-of-year, cyclic hour/day-of-week, and trend days. Weather: temperature, rain, humidity, wind, rain class, missingness, forecast flag. Events: event count, in-window flag, event-type flags, hours to start, hours since end. All are available at forecast time from the supplied test timestamp, weather forecast, and event calendar. `avg_fare_birr`, `avg_wait_min`, and `active_drivers` are not model features because they are train-only operational outcomes.

## A7–A8. Integrity and master tables
Generated files: `data/processed/master_train.csv`, `master_test.csv`, `data_dictionary_master.csv`; `reports/A1_cleaning_log.csv`, `A_join_audit.json`, `A7_integrity_checks.csv`. Check every PASS/FAIL before submitting. A failed assertion should be investigated, not ignored.